# COBRA local video analysis

Use this notebook to validate COBRA locally on an MP4 file with real Azure Speech and Azure OpenAI services. The notebook does not stub model responses.


## Prerequisites

- Install FFmpeg and ensure `ffmpeg` and `ffprobe` are available on `PATH`.
- Copy `sample.env` to `.env` and fill the required values.
- Authenticate to the Azure tenant that contains the configured resources before running the notebook.
- For keyless Speech transcription, set `AZURE_SPEECH_RESOURCE_ID`; the Speech SDK requires it for Entra ID tokens.


In [ ]:
%pip install -e ../.


## Required environment values

The local keyless path uses these `.env` values:

```text
AZURE_OPENAI_GPT_VISION_ENDPOINT="https://<resource>.cognitiveservices.azure.com/"
AZURE_OPENAI_GPT_VISION_API_KEY=""
AZURE_OPENAI_GPT_VISION_API_VERSION="2025-04-01-preview"
AZURE_OPENAI_GPT_VISION_DEPLOYMENT="<deployment-name>"

AZURE_SPEECH_REGION="<region>"
AZURE_SPEECH_USE_MANAGED_IDENTITY="true"
AZURE_SPEECH_RESOURCE_ID="/subscriptions/<sub>/resourceGroups/<rg>/providers/Microsoft.CognitiveServices/accounts/<resource>"
```

Storage and Search settings are optional for local-only validation. `VideoClient(upload_to_azure=False)` keeps local runs from uploading artifacts.


In [ ]:
from pathlib import Path

VIDEO_PATH = Path(r"C:\\path\\to\\your\\video.mp4")
OUTPUT_DIR = Path("../outputs") / VIDEO_PATH.stem

SEGMENT_LENGTH_SECONDS = 10
FRAMES_PER_SECOND = 0.5
GENERATE_TRANSCRIPTS = True

assert VIDEO_PATH.is_file(), f"Video file not found: {VIDEO_PATH}"


## Preprocess the video

This extracts frames, detects/extracts audio, converts audio to Speech-compatible WAV/PCM, and optionally generates a transcript.


In [ ]:
from cobrapy import VideoClient

client = VideoClient(video_path=str(VIDEO_PATH), upload_to_azure=False)
manifest_path = client.preprocess_video(
    output_directory=str(OUTPUT_DIR),
    segment_length=SEGMENT_LENGTH_SECONDS,
    fps=FRAMES_PER_SECOND,
    generate_transcripts_flag=GENERATE_TRANSCRIPTS,
    max_workers=1,
    overwrite_output=True,
)
manifest_path


In [ ]:
transcript = client.manifest.audio_transcription
print(f"segments: {len(client.manifest.segments)}")
print(f"audio found: {client.manifest.source_video.audio_found}")
print(f"source audio: {client.manifest.source_audio.path}")
print(f"transcript segments: {len(transcript.segments) if transcript else 0}")
print((transcript.text if transcript else "")[:500])


## Run Action Summary analysis

This sends the generated segment prompts and frames to the configured Azure OpenAI deployment.


In [ ]:
from cobrapy.analysis import ActionSummary

result = client.analyzer.analyze_video(
    analysis_config=ActionSummary(),
    run_async=False,
    reprocess_segments=False,
)
client.analyzer.latest_output_path


In [ ]:
from pprint import pprint

print(f"analysis entries: {len(result) if isinstance(result, list) else 'n/a'}")
pprint(result[:2] if isinstance(result, list) else result)


## Command-line alternative

The same flow can be run without Jupyter:

```powershell
python scripts\run_local_video_analysis.py "C:\path\to\video.mp4" --output-dir outputs\my-video --segment-length 10 --fps 0.5
```
